# Comparación de modelos — #22
Se leen automáticamente todos los `docs/results/*_comparison.csv`: cada modelo aporta sus filas con las columnas del contrato común. Los archivos de trials no se incluyen. La columna `source_csv` identifica el origen de cada fila; referencias Dummy repetidas se conservan explícitamente, sin deduplicar configuraciones de forma silenciosa.

La selección se basa en la media de F1 macro de validación, no en el F1 agregado OOF. Los informes OOF describen train y no son nested CV. Se exige el mismo snapshot, hash de datos y configuración de CV, además de scores válidos y esquema común; si hay archivos incompatibles, se corrigen antes de comparar.

La tabla no incluye resultados del test reservado ni selecciona automáticamente un champion. El champion y su justificación corresponden a #22, una vez disponibles los resultados comparables de los cuatro modelos. Solo ese champion se evalúa en test. La baseline tiene exposición histórica de esa partición; el equipo debe considerarla al interpretar una evaluación final.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
RESULT_COLUMNS = [
    'model', 'cv_train_f1_macro', 'cv_validation_f1_macro', 'cv_validation_std',
    'gap', 'gap_under_005', 'cv_fit_seconds', 'evaluation_set', 'oof_accuracy',
    'oof_f1_macro', 'oof_extreme_error_rate', 'oof_seconds', 'search_seconds',
    'params', 'cv', 'snapshot', 'code_commit', 'raw_sha256',
]
paths = sorted((ROOT / 'docs/results').glob('*_comparison.csv'))
if not paths:
    raise FileNotFoundError('No hay archivos docs/results/*_comparison.csv para comparar')
frames = []
expected_columns = None
for path in paths:
    frame = pd.read_csv(path)
    held_out_columns = [c for c in frame.columns if c.startswith('test_')]
    if held_out_columns:
        raise ValueError(f'{path.name}: columnas de test reservadas no permitidas: {held_out_columns}')
    missing = set(RESULT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name}: faltan columnas comunes: {sorted(missing)}')
    if 'source_csv' in frame.columns:
        raise ValueError(f'{path.name}: source_csv es una columna reservada para el origen')
    if expected_columns is None:
        expected_columns = set(frame.columns)
    elif set(frame.columns) != expected_columns:
        raise ValueError(f'{path.name}: las columnas deben coincidir con el resto de CSV de comparación')
    if frame.empty or frame['model'].isna().any() or frame['model'].astype(str).str.strip().eq('').any():
        raise ValueError(f'{path.name}: se necesita al menos una fila con nombre de modelo')
    if not frame['evaluation_set'].eq('train_oof').all():
        raise ValueError(f'{path.name}: los diagnósticos deben proceder de train_oof')
    for column in ['cv_train_f1_macro', 'cv_validation_f1_macro', 'cv_validation_std',
                   'oof_accuracy', 'oof_f1_macro', 'oof_extreme_error_rate']:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
        if not frame[column].between(0, 1).all():
            raise ValueError(f'{path.name}: {column} debe ser finito, no nulo y estar entre 0 y 1')
    frame['gap'] = pd.to_numeric(frame['gap'], errors='raise')
    if not np.isfinite(frame['gap']).all() or not np.allclose(
        frame['gap'], frame['cv_train_f1_macro'] - frame['cv_validation_f1_macro'], atol=1e-6,
    ):
        raise ValueError(f'{path.name}: gap no coincide con media train menos media validación')
    gap_flag = frame['gap_under_005'].map({True: True, False: False, 'True': True, 'False': False})
    if gap_flag.isna().any() or not gap_flag.eq(frame['gap'].lt(0.05)).all():
        raise ValueError(f'{path.name}: gap_under_005 no coincide con gap < 0.05')
    frames.append(frame.assign(source_csv=path.name))

results = pd.concat(frames, ignore_index=True)
for column in ['snapshot', 'raw_sha256', 'cv']:
    if results[column].isna().any() or results[column].nunique(dropna=False) != 1:
        raise ValueError(f'CSV no comparables: {column} debe ser el mismo y no nulo en todas las filas')
results = results.sort_values('cv_validation_f1_macro', ascending=False, kind='stable').reset_index(drop=True)
print(f'{len(paths)} archivos de comparación, {len(results)} filas. Orden por F1 macro medio de validación.')
print('La tabla no declara un champion ni evalúa el test reservado.')
display(results)


1 archivos de comparación, 2 filas. Orden por F1 macro medio de validación.
La tabla no declara un champion ni evalúa el test reservado.


,model,cv_train_f1_macro,cv_validation_f1_macro,cv_validation_std,gap,gap_under_005,cv_fit_seconds,evaluation_set,oof_accuracy,oof_f1_macro,oof_extreme_error_rate,oof_seconds,search_seconds,params,cv,snapshot,code_commit,raw_sha256,source_csv
0,LogisticRegression,0.739585,0.729123,0.006091,1.046206e-02,True,0.668796,train_oof,0.729679,0.729183,0.012490,0.650682,20.19471,"{""C"": 0.5971896122019431, ""class_weight"": null}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,5be5a63fcec6e779b070efb7d08027f5534c9bb3,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,logreg_comparison.csv
1,DummyClassifier,0.166791,0.166791,0.000068,8.618269e-09,True,0.236735,train_oof,0.333664,0.166791,0.333366,0.296802,0.00000,"{""strategy"": ""most_frequent""}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,5be5a63fcec6e779b070efb7d08027f5534c9bb3,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,logreg_comparison.csv
